# Donation-Intent Classifier Training — Fine-Tuned Encoder (RoBERTa vs DeBERTa-v3)

This notebook is the **classifier-training** counterpart to the earlier
LLM-as-annotator notebook. Instead of prompting a generative model, it
**fine-tunes a text encoder** (RoBERTa-base and DeBERTa-v3-base, compared
head-to-head) with two output heads:

- a **binary head** for `binary_label` (`yes` / `no`)
- a **three-way head** for `modifier` (`none` / `deferred` / `conditional`)

Both heads sit on the same pooled `[CLS]` representation and are trained
jointly (multi-task loss = binary CE + modifier CE). The `modifier` head is
architecturally secondary to `binary_label` per the task spec (its output is
only meaningful once `binary_label` is decided), but for a clean fixed-shape
model interface **both heads are computed for every example**; the
`modifier` loss/metrics are reported both overall and specifically among
`binary_label == "yes"` rows (the population the task spec conditions on),
so the analysis stays faithful to the "conditioned on `binary_label=yes`"
framing.

**Data.** Full manually-labeled dataset of **1017** conversations
(`conversation_id, dialogue_id, dialogue_text, binary_label, modifier`).
Unlike the earlier 250-row 5-annotator sample, this dataset has **no
partition/annotator column**, so there is no partition-specific
drift check in this run. A blank/NaN `modifier` value is treated as `"none"`.
Input discovery reuses the same fuzzy, case-insensitive filename search as
the earlier notebook, so it picks the CSV up automatically once added as a
Kaggle input — no path edits needed.

**Split.** Stratified **70/15/15** train/validation/test split, stratified by
the joint `binary_label`×`modifier` distribution so all three splits keep a
similar label mix.

**Class imbalance.** Handled two ways simultaneously, per the observed
imbalance (Table II):
1. **Class weights** — inverse-frequency weights fed into both heads' loss
   functions.
2. **Oversampling** — a `WeightedRandomSampler` over the *training* split
   only (val/test stay untouched, so evaluation numbers stay honest) that
   oversamples minority joint-label combinations.

A **majority-class baseline** is reported alongside the trained models for
both heads, as required by the task spec.

**Evaluation.** Accuracy + F1 for the binary head, macro-F1 for the modifier
head, plus the majority-class baseline, for **both encoders**, on the held-out
test split.


## 0b. Install / upgrade dependencies

Installs required packages from PyPI. Running in **online mode** on Kaggle T4.
The `--no-deps` guard on `accelerate` prevents silently downgrading a
GPU-matched `torch` build already present on the Kaggle image.

In [ ]:
# =============================================================
# 0b. INSTALL / UPGRADE DEPENDENCIES (online mode)
# =============================================================
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers>=4.44.0,<4.47.0", "sentencepiece", "tqdm",
    "seaborn", "scikit-learn",
], check=True)
# --no-deps: keep the GPU-matched torch already on the Kaggle image
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "--no-deps", "accelerate>=0.34.0",
], check=True)
print("All required Python packages installed successfully.")


## 1. Imports & global config

In [ ]:
# =============================================================
# 1. IMPORTS & GLOBAL CONFIG
# =============================================================
import os, re, json, glob, gc, time, warnings, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                              classification_report, confusion_matrix)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
print(f"torch version: {torch.__version__} (built for CUDA {torch.version.cuda})")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ---- CONFIG: the only block you should need to touch -----------------
CONFIG = {
    # Same fuzzy recursive search-root convention as the LLM-annotator
    # notebook, so this works unmodified whether the new 1017-row CSV is
    # added as a Kaggle Dataset input or dropped anywhere under these roots.
    "search_roots": ["/kaggle/input", "/kaggle/working", "/workspace", "/data", "."],

    "out_dir": "/kaggle/working" if os.path.isdir("/kaggle/working") else "./outputs",

    # Column-name mapping. If the CSV uses different headers, only this
    # dict needs editing -- nothing downstream references raw column names.
    "column_map": {
        "text": "dialogue_text",
        "binary_label": "binary_label",
        "modifier": "modifier",
        # NOTE: the 1017-row dataset has no annotator/partition column (unlike
        # the earlier 5-annotator CSVs), so there is no partition-drift check
        # in this run -- splitting/stratification below is by label only.
    },

    "binary_classes": ["no", "yes"],
    "modifier_classes": ["none", "deferred", "conditional"],

    # Three encoders benchmarked head-to-head.
    # Per-entry "lr" and "batch_size" override the global defaults;
    # DeBERTa-v3 needs a lower LR (1e-5) and smaller batch (8) for stability.
    "encoders": [
        {"name": "roberta-base",    "hf_id": "roberta-base"},
        {"name": "deberta-v3-base", "hf_id": "microsoft/deberta-v3-base","lr": 5e-6, "batch_size": 8, "warmup_ratio": 0.10},
        {"name": "todbert",         "hf_id": "TODBERT/TOD-BERT-JNT-V1"},
    ],

    "max_length": 256,
    "train_frac": 0.70,
    "val_frac": 0.15,
    "test_frac": 0.15,

    "batch_size": 16,
    "eval_batch_size": 32,
    "num_epochs": 25,
    "lr": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.06,
    "max_grad_norm": 1.0,

    # Multi-task loss weighting: binary head is the primary task per the
    # task spec ("conditioned on binary_label=yes"), modifier head is
    # secondary -- weighted down slightly so it doesn't dominate the shared
    # encoder gradient.
    "binary_loss_weight": 1.0,
    "modifier_loss_weight": 0.7,

    # Imbalance handling: inverse-frequency class weights in the loss.
    # Oversampling (WeightedRandomSampler) is DISABLED -- stacking it on
    # top of class weights multiplies their effects and over-rotates the
    # model toward minority classes, collapsing macro-F1 below baseline.
    "use_class_weights": True,
    "use_oversampling": False,

    # Speaker-aware encoding: add learned role embeddings (Persuader/
    # Persuadee/special) to per-token hidden states before pooling.
    # Role IDs are detected from existing [Persuader]/[Persuadee] markers
    # in the dialogue text -- no hardcoding, no separate classification head.
    "use_speaker_roles": True,

    # ---- Data augmentation (EDA) -----------------------------------------
    # Applied to the TRAINING split only, AFTER the stratified split, so
    # nothing synthetic ever leaks into val/test. Targets the classes that
    # are structurally starved of examples: binary="no" (mild boost) and
    # modifier="deferred"/"conditional" among binary="yes" rows (heavy
    # boost -- conditional has only ~9-13 real training examples).
    "use_augmentation": True,
    "aug_alpha": 0.15,          # fraction of eligible words perturbed per op
    "aug_num_conditional": 6,   # augmented copies per (yes, conditional) row
    "aug_num_deferred": 2,      # augmented copies per (yes, deferred) row
    "aug_num_no": 1,            # augmented copies per (binary=no) row

    # Label smoothing on both classification heads. Softens cross-entropy
    # so the model can't drive full confidence into one class early and get
    # stuck there -- this is what happened to DeBERTa's modifier head last
    # run: it collapsed to predicting "none" for 100% of rows within a
    # couple epochs and never recovered.
    "label_smoothing": 0.1,

    "early_stopping_patience": 5,   # epochs without val macro-F1 improvement
    "checkpoint_dir_name": "checkpoints",
}

os.makedirs(CONFIG["out_dir"], exist_ok=True)
os.makedirs(os.path.join(CONFIG["out_dir"], CONFIG["checkpoint_dir_name"]), exist_ok=True)
print(json.dumps({k: v for k, v in CONFIG.items() if k != "encoders"}, indent=2))
print("Encoders:", [m["hf_id"] for m in CONFIG["encoders"]])


## 2. Load the labeled dataset

Reuses the earlier notebook's robust, case-insensitive recursive filename
search (handles `Manual_Label__X.csv` vs `Manual Label - X.csv` vs whatever
naming the 1017-row dataset upload ends up with). If more than one matching
CSV is found (e.g. the new dataset is split into parts) they're concatenated;
duplicate `conversation_id`s are dropped, keeping the first occurrence.

In [ ]:
# =============================================================
# 2. DATA DISCOVERY & LOADING
# =============================================================

def find_files_ci(roots, must_contain_all, suffix):
    must_contain_all = [t.lower() for t in must_contain_all]
    suffix = suffix.lower()
    found = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _dirnames, filenames in os.walk(root):
            for fn in filenames:
                low = fn.lower()
                if low.endswith(suffix) and all(tok in low for tok in must_contain_all):
                    found.append(os.path.join(dirpath, fn))
    seen, unique = set(), []
    for f in found:
        if f not in seen:
            seen.add(f)
            unique.append(f)
    return unique

# Broad token so this matches whatever the new 1017-row dataset gets named
# (e.g. "labeled_dataset.csv", "manual_labels_full.csv", ...). Falls back to
# the narrower "manual"+"label" search used by the old notebook if nothing
# broader turns up.
csv_paths = find_files_ci(CONFIG["search_roots"], must_contain_all=["label"], suffix=".csv")
if not csv_paths:
    csv_paths = find_files_ci(CONFIG["search_roots"], must_contain_all=["manual"], suffix=".csv")

print(f"Found {len(csv_paths)} candidate CSV(s):")
for p in csv_paths:
    print(" -", p)

assert len(csv_paths) > 0, (
    "No labeled CSV found under " + str(CONFIG["search_roots"]) +
    ". Make sure the 1017-row dataset has been added as a Kaggle input "
    "(Notebook -> Add Input), or update CONFIG['search_roots']."
)

_dfs = [pd.read_csv(p) for p in csv_paths]
raw_df = pd.concat(_dfs, ignore_index=True) if len(_dfs) > 1 else _dfs[0]

cm = CONFIG["column_map"]
missing_cols = [c for c in cm.values() if c not in raw_df.columns]
assert not missing_cols, (
    f"Expected column(s) {missing_cols} not found in loaded CSV(s). "
    f"Columns present: {list(raw_df.columns)}. "
    "Update CONFIG['column_map'] to match the new dataset's headers."
)

df = raw_df.rename(columns={v: k for k, v in cm.items()})[list(cm.keys())].copy()

# De-dup on the original id column (conversation_id / dialogue_id / id),
# whichever is present, so a re-uploaded or overlapping CSV doesn't double-count rows.
orig_id_col = None
for cand in ["conversation_id", "dialogue_id", "id"]:
    if cand in raw_df.columns:
        orig_id_col = cand
        break
if orig_id_col is not None:
    df["_orig_id"] = raw_df[orig_id_col].values
    before = len(df)
    df = df.drop_duplicates(subset="_orig_id").drop(columns="_orig_id").reset_index(drop=True)
    if len(df) != before:
        print(f"Dropped {before - len(df)} duplicate row(s) by `{orig_id_col}`.")

df["text"] = df["text"].fillna("").astype(str)
df["binary_label"] = df["binary_label"].fillna("").astype(str).str.strip().str.lower()

# Empty/NaN modifier means "none" (confirmed). IMPORTANT: fillna BEFORE
# astype(str) -- pandas' astype(str) does not reliably stringify real NaN
# values on object-dtype columns (they can survive as float NaN), so
# checking for the string "nan" afterward would silently miss them.
df["modifier"] = df["modifier"].fillna("").astype(str).str.strip().str.lower()
df.loc[df["modifier"].isin(["", "nan", "none provided", "na"]), "modifier"] = "none"

bad_binary = ~df["binary_label"].isin(CONFIG["binary_classes"])
bad_modifier = ~df["modifier"].isin(CONFIG["modifier_classes"])
if bad_binary.any() or bad_modifier.any():
    print(f"WARNING: dropping {int((bad_binary | bad_modifier).sum())} row(s) with "
          f"out-of-vocabulary binary_label/modifier values.")
    print("  bad binary_label values:", sorted(df.loc[bad_binary, "binary_label"].unique()))
    print("  bad modifier values:", sorted(df.loc[bad_modifier, "modifier"].unique()))
    df = df.loc[~(bad_binary | bad_modifier)].reset_index(drop=True)

df["binary_id"] = df["binary_label"].map({c: i for i, c in enumerate(CONFIG["binary_classes"])})
df["modifier_id"] = df["modifier"].map({c: i for i, c in enumerate(CONFIG["modifier_classes"])})

print(f"\nLoaded {len(df)} labeled rows after cleaning.")
print("No partition/annotator column in this dataset -- splitting below is by label only.")
df.head()


## 2b. Class distribution & majority-class baseline

Reports the label imbalance (per the task spec: "class imbalance already
observed"), and computes the majority-class baseline accuracy/F1 for both
heads that the trained models are compared against.

In [ ]:
# =============================================================
# 2b. CLASS DISTRIBUTION & MAJORITY-CLASS BASELINE
# =============================================================
print("binary_label distribution:")
print(df["binary_label"].value_counts(), "\n")
print("modifier distribution:")
print(df["modifier"].value_counts(), "\n")
print("modifier distribution, restricted to binary_label == 'yes' rows "
      "(the population the modifier head is conditioned on):")
print(df.loc[df["binary_label"] == "yes", "modifier"].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(data=df, x="binary_label", order=CONFIG["binary_classes"], ax=axes[0])
axes[0].set_title("binary_label distribution")
sns.countplot(data=df, x="modifier", order=CONFIG["modifier_classes"], ax=axes[1])
axes[1].set_title("modifier distribution")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["out_dir"], "label_distribution.png"), dpi=150)
plt.show()

def majority_baseline(y_true_ids, n_classes):
    majority_class = Counter(y_true_ids).most_common(1)[0][0]
    y_pred = [majority_class] * len(y_true_ids)
    acc = accuracy_score(y_true_ids, y_pred)
    f1_macro = f1_score(y_true_ids, y_pred, average="macro", labels=list(range(n_classes)), zero_division=0)
    return {"majority_class": majority_class, "accuracy": acc, "macro_f1": f1_macro}

baseline_binary = majority_baseline(df["binary_id"].values, len(CONFIG["binary_classes"]))
baseline_modifier = majority_baseline(df["modifier_id"].values, len(CONFIG["modifier_classes"]))

print("\nMajority-class baseline (whole dataset, for reference -- the real "
      "reported baseline further down uses the TEST split only):")
print("  binary_label:", baseline_binary)
print("  modifier    :", baseline_modifier)


## 3. Stratified 70/15/15 train/val/test split

Stratified by the joint `binary_label`×`modifier` combination, so all three
splits keep a similar label mix (needed for honest evaluation). There is no
partition/annotator column in this 1017-row dataset (unlike the earlier
5-annotator CSVs), so there is no partition-based stratification and no
partition-specific drift check in this run -- if a partition-style column
becomes available later, section 3 and section 11 from the original design
can stratify/report on it directly.


In [ ]:
# =============================================================
# 3. STRATIFIED 70/15/15 SPLIT (by binary_label x modifier combination)
# =============================================================

df["_stratum"] = df["binary_label"] + "_" + df["modifier"]

# Collapse strata with <2 members into a single bucket, so stratify= never
# raises on a singleton combination.
strat_counts = df["_stratum"].value_counts()
rare = strat_counts[strat_counts < 2].index
df.loc[df["_stratum"].isin(rare), "_stratum"] = "_singleton_bucket"

train_df, temp_df = train_test_split(
    df, test_size=(CONFIG["val_frac"] + CONFIG["test_frac"]),
    stratify=df["_stratum"], random_state=SEED,
)
# re-derive rare strata on temp_df (some may now be too small) before the 2nd split
temp_counts = temp_df["_stratum"].value_counts()
rare2 = temp_counts[temp_counts < 2].index
temp_df = temp_df.copy()
temp_df.loc[temp_df["_stratum"].isin(rare2), "_stratum"] = "_singleton_bucket"

rel_test_size = CONFIG["test_frac"] / (CONFIG["val_frac"] + CONFIG["test_frac"])
val_df, test_df = train_test_split(
    temp_df, test_size=rel_test_size,
    stratify=temp_df["_stratum"], random_state=SEED,
)

for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name}: n={len(split)} ({len(split)/len(df):.1%})")

print("\nbinary_label composition per split:")
display(pd.concat({
    "train": train_df["binary_label"].value_counts(normalize=True),
    "val": val_df["binary_label"].value_counts(normalize=True),
    "test": test_df["binary_label"].value_counts(normalize=True),
}, axis=1))

print("\nmodifier composition per split:")
display(pd.concat({
    "train": train_df["modifier"].value_counts(normalize=True),
    "val": val_df["modifier"].value_counts(normalize=True),
    "test": test_df["modifier"].value_counts(normalize=True),
}, axis=1))

for split in (train_df, val_df, test_df):
    split.drop(columns=["_stratum"], inplace=True)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)


## 3b. Data Augmentation (EDA) --- TRAIN SPLIT ONLY

The modifier head is starved of minority-class examples: after the 70/15/15
split, training has roughly **9-13** `conditional` and **~48** `deferred`
examples (out of 738 `yes` rows) -- nowhere near enough for a 125M-parameter
encoder to learn a reliable decision boundary. That's why every encoder's
`conditional` F1 is 0.00 and why DeBERTa's modifier head collapsed entirely
to predicting the majority class last run.

This cell applies **Easy Data Augmentation (EDA)** -- synonym replacement,
random swap, random deletion, random insertion -- to `train_df` only, run
*after* the stratified split so no synthetic text can leak into val/test.

**Label-safety guard:** plain EDA can silently destroy the very signal that
distinguishes `deferred`/`conditional` from `none` if it happens to replace
or delete words like "if", "after", "once", "promise", "later". A protected
keyword list excludes these (and the `[Persuader]`/`[Persuadee]` speaker
markers) from every perturbation, so augmented copies keep the linguistic
cue that justifies their label.

Augmentation intensity is scaled to how starved each class is:
- `conditional` rows (binary=yes) -> 6 augmented copies each (most starved)
- `deferred` rows (binary=yes) -> 2 augmented copies each
- `no` rows (binary=no) -> 1 augmented copy each (mild boost)
- `none` rows (binary=yes, modifier=none) -> **not augmented** (already majority)


In [ ]:
# =============================================================
# 3b. EDA AUGMENTATION (train split only, label-safety-guarded)
# =============================================================
import random as _random

# Words that carry the actual modifier signal (conditional/deferred cues) or
# negation/donation semantics. NEVER replaced, deleted, or treated as
# swappable-away -- doing so would silently relabel an example.
PROTECTED_WORDS = {
    "if", "unless", "provided", "as", "long", "condition", "conditional",
    "after", "once", "later", "when", "then", "next", "eventually",
    "promise", "will", "would", "could", "might", "may", "maybe",
    "paycheck", "payday", "month", "week", "tomorrow", "soon",
    "not", "no", "never", "don't", "dont", "won't", "wont", "can't", "cant",
    "donate", "donation", "give", "money", "charity", "pledge",
}
SPEAKER_MARKERS = {"[persuader]", "[persuadee]"}

# Small hand-built synonym table (domain-safe, offline, no external
# corpus download needed -- avoids depending on nltk/wordnet availability
# on the Kaggle runtime, and avoids WordNet occasionally proposing a
# synonym for a *protected* sense of a word we've already excluded above).
_SYNONYMS = {
    "good": ["nice", "great", "fine", "decent"],
    "really": ["truly", "genuinely", "honestly"],
    "think": ["believe", "feel", "figure"],
    "want": ["would like", "wish", "hope"],
    "help": ["assist", "support", "aid"],
    "people": ["folks", "individuals", "persons"],
    "important": ["significant", "vital", "essential"],
    "cause": ["mission", "campaign", "effort"],
    "understand": ["see", "get", "realize"],
    "sure": ["certain", "confident", "positive"],
    "sorry": ["apologies", "regret", "my bad"],
    "okay": ["alright", "fine", "sure"],
    "thanks": ["thank you", "appreciate it", "cheers"],
    "great": ["awesome", "wonderful", "fantastic"],
    "kids": ["children", "youth", "young ones"],
    "families": ["households", "homes"],
    "problem": ["issue", "trouble", "difficulty"],
    "talk": ["chat", "speak", "discuss"],
    "guess": ["suppose", "reckon", "figure"],
    "job": ["work", "career", "position"],
    "busy": ["swamped", "occupied", "tied up"],
}

def _get_synonym(word):
    lw = word.lower()
    if lw in _SYNONYMS:
        return _random.choice(_SYNONYMS[lw])
    return None

def _tokenize_protecting_markers(text):
    """Split on whitespace but keep [Persuader]/[Persuadee] markers intact
    as single protected tokens (they may be glued to punctuation)."""
    return text.split(" ")

def _is_locked(tok):
    low = tok.strip(".,!?;:\"'").lower()
    return low in PROTECTED_WORDS or low in SPEAKER_MARKERS or tok.lower() in SPEAKER_MARKERS

def eda_synonym_replacement(words, n):
    new_words = words.copy()
    candidates = [i for i, w in enumerate(new_words) if not _is_locked(w) and _get_synonym(w)]
    _random.shuffle(candidates)
    replaced = 0
    for idx in candidates:
        syn = _get_synonym(new_words[idx])
        if syn:
            new_words[idx] = syn
            replaced += 1
        if replaced >= n:
            break
    return new_words

def eda_random_deletion(words, p):
    if len(words) <= 3:
        return words.copy()
    new_words = []
    for w in words:
        if _is_locked(w):
            new_words.append(w)
        elif _random.random() > p:
            new_words.append(w)
    if len(new_words) == 0:
        return [words[_random.randrange(len(words))]]
    return new_words

def eda_random_swap(words, n):
    new_words = words.copy()
    swappable = [i for i, w in enumerate(new_words) if not _is_locked(w)]
    for _ in range(n):
        if len(swappable) < 2:
            break
        i, j = _random.sample(swappable, 2)
        new_words[i], new_words[j] = new_words[j], new_words[i]
    return new_words

def eda_random_insertion(words, n):
    new_words = words.copy()
    for _ in range(n):
        source_candidates = [w for w in new_words if not _is_locked(w) and _get_synonym(w)]
        if not source_candidates:
            break
        w = _random.choice(source_candidates)
        syn = _get_synonym(w)
        insert_pos = _random.randrange(len(new_words) + 1)
        new_words.insert(insert_pos, syn)
    return new_words

def eda_augment_one(text, alpha=0.15, seed=None):
    """Return ONE augmented variant of `text`, applying a random subset of
    the 4 EDA operations at intensity `alpha`, with speaker markers and
    intent-bearing words protected throughout."""
    if seed is not None:
        _random.seed(seed)
    words = _tokenize_protecting_markers(text)
    n_ops = max(1, int(alpha * len(words)))

    op = _random.choice(["synonym", "swap", "delete", "insert"])
    if op == "synonym":
        words = eda_synonym_replacement(words, n_ops)
    elif op == "swap":
        words = eda_random_swap(words, n_ops)
    elif op == "delete":
        words = eda_random_deletion(words, p=alpha)
    else:
        words = eda_random_insertion(words, n_ops)

    return " ".join(words)

def augment_dataframe_for_minority_classes(df, config, seed=42):
    """Build augmented copies of starved-class rows and append them to df.
    Only ever called on train_df, after the split. Returns a NEW dataframe
    (original df rows are untouched, so caller can still trace provenance)."""
    _random.seed(seed)
    rows_to_add = []

    yes_idx = config["binary_classes"].index("yes")
    no_idx = config["binary_classes"].index("no")
    cond_idx = config["modifier_classes"].index("conditional")
    def_idx = config["modifier_classes"].index("deferred")

    plan = [
        (df["binary_id"] == yes_idx) & (df["modifier_id"] == cond_idx), config["aug_num_conditional"],
        (df["binary_id"] == yes_idx) & (df["modifier_id"] == def_idx),  config["aug_num_deferred"],
        (df["binary_id"] == no_idx),                                   config["aug_num_no"],
    ]
    # plan is a flat [mask, n, mask, n, mask, n] list -- pair them up
    for mask, n_copies in zip(plan[0::2], plan[1::2]):
        subset = df.loc[mask]
        for _, row in subset.iterrows():
            for k in range(n_copies):
                aug_text = eda_augment_one(row["text"], alpha=config["aug_alpha"],
                                            seed=hash((row["text"], k)) % (2**31))
                new_row = row.copy()
                new_row["text"] = aug_text
                rows_to_add.append(new_row)

    if not rows_to_add:
        return df.reset_index(drop=True)

    aug_df = pd.DataFrame(rows_to_add)
    out = pd.concat([df, aug_df], ignore_index=True)
    return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)


if CONFIG.get("use_augmentation", False):
    _before_n = len(train_df)
    _before_dist = train_df.loc[train_df["binary_label"] == "yes", "modifier"].value_counts()

    train_df = augment_dataframe_for_minority_classes(train_df, CONFIG, seed=SEED)

    _after_n = len(train_df)
    _after_dist = train_df.loc[train_df["binary_label"] == "yes", "modifier"].value_counts()

    print(f"Train split size: {_before_n} -> {_after_n} rows after EDA augmentation")
    print("\nmodifier distribution among binary=yes TRAIN rows, before -> after:")
    display(pd.concat({"before": _before_dist, "after": _after_dist}, axis=1).fillna(0).astype(int))
    print("\nbinary_label distribution after augmentation:")
    print(train_df["binary_label"].value_counts())
else:
    print("Augmentation disabled (CONFIG['use_augmentation'] = False).")


## 4. Dataset & tokenization

One `Dataset` class shared across both encoders (only the tokenizer passed in
differs). Tokenization happens lazily per-batch via the `collate_fn` so the
same class works for both encoders without re-instantiating per model.

In [ ]:
# =============================================================
# 4. TORCH DATASET + SPEAKER-ROLE ID COMPUTATION
# =============================================================
import re

# Role constants (used only in preprocessing — NOT hardcoded in the model).
_SPEAKER_ROLE_MAP = {"[Persuader]": 0, "[Persuadee]": 1}
_SPEAKER_PATTERN  = re.compile(r'\[Persuader\]|\[Persuadee\]')
ROLE_PERSUADER = 0   # tokens belonging to Persuader turns
ROLE_PERSUADEE = 1   # tokens belonging to Persuadee turns
ROLE_SPECIAL   = 2   # CLS, SEP, PAD, or tokens before the first marker


class DonationIntentDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe["text"].tolist()
        self.binary_ids = dataframe["binary_id"].tolist()
        self.modifier_ids = dataframe["modifier_id"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "text": self.texts[idx],
            "binary_id": self.binary_ids[idx],
            "modifier_id": self.modifier_ids[idx],
        }


def compute_batch_role_ids(texts, offset_mapping):
    """
    Assign speaker role IDs to each token position using character offsets.

    This is the preferred approach because it works correctly for all
    sub-word tokenizers (BPE for RoBERTa/TOD-BERT, SentencePiece for
    DeBERTa-v3) without any segment-boundary tokenization artefacts.

    Algorithm
    ---------
    For each text in the batch:
      1. Find all [Persuader] / [Persuadee] markers via regex (character positions).
      2. Sort markers by position → role_boundaries list.
      3. For each token's (char_start, char_end) offset:
         - (0,0) → special token → role 2
         - else  → walk role_boundaries to find the most recent marker
                   whose start ≤ char_start → assign that role.

    This does NOT classify or predict roles — it reads structural markers
    that are already in the data and propagates them token-by-token.

    Parameters
    ----------
    texts          : list[str]          -- raw dialogue texts in this batch
    offset_mapping : Tensor[B, L, 2]   -- per-token (char_start, char_end)
                     returned by tokenizer(..., return_offsets_mapping=True)

    Returns
    -------
    Tensor[B, L] of dtype long, values in {0, 1, 2}
    """
    batch_role_ids = []

    for text, token_offsets in zip(texts, offset_mapping.tolist()):
        # Build sorted list of (position, role) for every marker in this text.
        boundaries = sorted(
            [(m.start(), _SPEAKER_ROLE_MAP[m.group()])
             for m in _SPEAKER_PATTERN.finditer(text)],
            key=lambda x: x[0],
        )

        role_ids = []
        for char_start, char_end in token_offsets:
            if char_start == 0 and char_end == 0:
                role_ids.append(ROLE_SPECIAL)   # CLS / SEP / PAD
                continue

            # Walk boundaries: the last boundary whose position ≤ char_start
            # is the active role for this token.
            current_role = ROLE_SPECIAL  # default before any marker
            for bpos, brole in boundaries:
                if bpos <= char_start:
                    current_role = brole
                else:
                    break   # boundaries are sorted; no need to look further
            role_ids.append(current_role)

        batch_role_ids.append(role_ids)

    return torch.tensor(batch_role_ids, dtype=torch.long)


def make_collate_fn(tokenizer, max_length, use_speaker_roles=False):
    """
    Returns a DataLoader collate function.

    When use_speaker_roles=True the function additionally asks the tokenizer
    for character-offset mappings and uses them to produce a 'role_ids' tensor
    for each batch.  Falls back gracefully (no role IDs, a warning printed) if
    the tokenizer does not support offset mapping.
    """
    def collate_fn(batch):
        texts = [b["text"] for b in batch]

        if use_speaker_roles:
            try:
                enc = tokenizer(
                    texts, padding=True, truncation=True,
                    max_length=max_length, return_tensors="pt",
                    return_offsets_mapping=True,
                )
                offset_mapping = enc.pop("offset_mapping")
                enc["role_ids"] = compute_batch_role_ids(texts, offset_mapping)
            except Exception as exc:
                print(f"  [WARNING] offset_mapping not available, "
                      f"speaker roles disabled for this batch: {exc}")
                enc = tokenizer(
                    texts, padding=True, truncation=True,
                    max_length=max_length, return_tensors="pt",
                )
        else:
            enc = tokenizer(
                texts, padding=True, truncation=True,
                max_length=max_length, return_tensors="pt",
            )

        enc["binary_labels"]   = torch.tensor([b["binary_id"]   for b in batch], dtype=torch.long)
        enc["modifier_labels"] = torch.tensor([b["modifier_id"] for b in batch], dtype=torch.long)
        return enc
    return collate_fn


def make_oversampling_sampler(dataframe):
    # Oversample by the joint (binary_label, modifier) combination on the
    # TRAINING split only -- inverse-frequency sample weights fed to
    # WeightedRandomSampler(replacement=True).
    joint   = dataframe["binary_label"] + "_" + dataframe["modifier"]
    counts  = joint.value_counts()
    weights = joint.map(lambda k: 1.0 / counts[k]).values
    return WeightedRandomSampler(
        weights=torch.as_tensor(weights, dtype=torch.double),
        num_samples=len(dataframe), replacement=True,
    )


## 5. Two-head model architecture

A shared transformer encoder with two independent linear heads on the pooled
`[CLS]`/first-token representation:
- `binary_head`: 2-way (`binary_label`)
- `modifier_head`: 3-way (`modifier`)

Both heads are always computed (fixed-shape output, simpler batching); the
task's "conditioned on `binary_label=yes`" framing is applied at the
**loss-weighting and evaluation** level (see below), not by structurally
gating the forward pass, since gating on a *predicted* label during training
would make the modifier head's gradient depend on the (initially noisy)
binary head and destabilize joint training. `AutoModel` (not
`AutoModelForSequenceClassification`) is used so the same custom two-head
class works identically for RoBERTa and DeBERTa-v3 without subclassing per
architecture.

In [ ]:
# =============================================================
# 5. SPEAKER-AWARE TWO-HEAD MODEL
# =============================================================

class SpeakerAwareTwoHeadClassifier(nn.Module):
    """
    Multi-task classifier with optional speaker-role embedding.

    Architecture
    ------------
    1. Shared transformer encoder (RoBERTa / DeBERTa-v3 / TOD-BERT via AutoModel).
    2. Speaker-role embedding [optional, enabled by use_speaker_roles=True]:
       - nn.Embedding(3, hidden_size) maps each token's role ID to a
         hidden-size vector, analogous to BERT's segment (A/B) embeddings.
       - Added element-wise to the encoder's *last hidden states* BEFORE
         mean-pooling, so the pooled representation carries role-aware context.
       - Initialised with small std (0.02) so training starts near the
         non-role-aware baseline; role corrections are learned incrementally.
    3. Mean-pool over non-padding token hidden states (attention_mask == 1).
       Both RoBERTa and DeBERTa-v3 use mean-pooling here (DeBERTa-v3's
       AutoModel does not expose a pooler_output).
    4. Dropout + two independent linear classification heads:
       - binary_head  : 2-class (no / yes)
       - modifier_head: 3-class (none / deferred / conditional)

    Role IDs
    --------
    Role 0 = Persuader tokens
    Role 1 = Persuadee tokens
    Role 2 = special / padding / tokens before the first marker

    Role IDs are computed by compute_batch_role_ids() from character offsets
    returned by the tokenizer.  The model itself has NO hardcoded role strings
    and NO separate role-classification head — the embedding is purely a
    learnable bias added to existing hidden states.
    """

    NUM_ROLES = 3  # Persuader / Persuadee / special-padding-unknown

    def __init__(self, hf_id, n_binary, n_modifier, dropout=0.1,
                 use_speaker_roles=False):
        super().__init__()
        self.use_speaker_roles = use_speaker_roles
        self.encoder = AutoModel.from_pretrained(hf_id)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)

        if use_speaker_roles:
            self.role_embedding = nn.Embedding(self.NUM_ROLES, hidden)
            nn.init.normal_(self.role_embedding.weight, std=0.02)
        else:
            self.role_embedding = None

        self.binary_head   = nn.Linear(hidden, n_binary)
        self.modifier_head = nn.Linear(hidden, n_modifier)

    def _mean_pool(self, last_hidden_state, attention_mask):
        """Masked mean-pool over token dimension."""
        mask    = attention_mask.unsqueeze(-1).float()
        summed  = (last_hidden_state * mask).sum(dim=1)
        counts  = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    def forward(self, input_ids, attention_mask, token_type_ids=None,
                role_ids=None, **_):
        import inspect
        enc_kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        fwd_params = inspect.signature(self.encoder.forward).parameters
        if token_type_ids is not None and "token_type_ids" in fwd_params:
            enc_kwargs["token_type_ids"] = token_type_ids

        outputs     = self.encoder(**enc_kwargs)
        last_hidden = outputs.last_hidden_state          # [B, L, H]

        # ── Speaker-role correction (before pooling) ──────────────────────
        # Adding role embeddings here lets the mean-pool aggregate a
        # representation that distinguishes Persuader vs Persuadee context,
        # which is the key signal for predicting donation intent.
        if self.use_speaker_roles and role_ids is not None:
            role_embeds = self.role_embedding(role_ids)  # [B, L, H]
            last_hidden = last_hidden + role_embeds

        pooled = self.dropout(self._mean_pool(last_hidden, attention_mask))
        return self.binary_head(pooled), self.modifier_head(pooled)


## 6. Class weights (inverse-frequency)

Computed on the **training split only** (never val/test). Binary class weights
use the full training split. Modifier class weights use **only `binary_label="yes"`
rows** to match the conditioned evaluation metric
(`modifier_macro_f1 | binary_label=yes`).

Oversampling (`WeightedRandomSampler`) is **disabled** -- stacking it on top of
inverse-frequency class weights multiplies their effects and over-rotates the
model toward minority classes, collapsing macro-F1 below the majority baseline.

In [ ]:
# =============================================================
# 6. INVERSE-FREQUENCY CLASS WEIGHTS (from TRAIN split only)
# =============================================================

def inverse_freq_weights(ids, n_classes):
    counts = np.bincount(ids, minlength=n_classes).astype(float)
    counts[counts == 0] = 1.0  # avoid div-by-zero for an absent class in train
    weights = counts.sum() / (n_classes * counts)
    return torch.tensor(weights, dtype=torch.float)

binary_class_weights = inverse_freq_weights(train_df["binary_id"].values, len(CONFIG["binary_classes"]))

# Modifier weights from yes-only training rows.
# The modifier task is conditioned on binary_label="yes": those rows have
# distribution (none ~79%, deferred ~18%, conditional ~3%), which differs
# from the full-dataset mix (85.6% / 12.4% / 2.0%). Using all rows would
# produce biased weights that do not match the evaluation distribution.
train_yes = train_df[train_df["binary_label"] == "yes"].reset_index(drop=True)
modifier_class_weights = inverse_freq_weights(
    train_yes["modifier_id"].values, len(CONFIG["modifier_classes"]))

print("binary_label class weights:", dict(zip(CONFIG["binary_classes"], binary_class_weights.tolist())))
print("modifier class weights (yes-only rows):", dict(zip(CONFIG["modifier_classes"], modifier_class_weights.tolist())))
print(f"  (computed from {len(train_yes)} yes-only / {len(train_df)} total training rows)")


## 7. Training loop (shared across all encoders)

Joint loss = `binary_loss_weight * CE(binary) + modifier_loss_weight * CE(modifier|yes)`,
where the modifier CE is computed **only on `binary_label="yes"` rows per batch**
(all `no` rows have `modifier="none"` by definition -- including them dilutes the
modifier signal and creates a mismatch between training and evaluation distributions).
Early stopping uses binary F1 + yes-conditioned modifier macro-F1, matching the
task evaluation metric.

In [ ]:
# =============================================================
# 7. TRAIN / EVAL FUNCTIONS (with threshold tuning)
# =============================================================

def evaluate(model, loader, device, binary_threshold=0.5):
    """
    Evaluate model on a DataLoader.

    binary_threshold : float
        P(yes) >= binary_threshold -> predict 'yes'.
        Default 0.5 is standard argmax. Tuned values < 0.5 recover 'no'
        recall without retraining, correcting the yes-bias caused by the
        72.5 % majority class.
    """
    model.eval()
    all_binary_true, all_binary_pred = [], []
    all_binary_probs = []
    all_modifier_true, all_modifier_pred = [], []
    total_loss = 0.0
    n_batches  = 0
    yes_idx    = CONFIG["binary_classes"].index("yes")

    bce = nn.CrossEntropyLoss(
        weight=binary_class_weights.to(device) if CONFIG["use_class_weights"] else None,
        label_smoothing=CONFIG.get("label_smoothing", 0.0))
    mce = nn.CrossEntropyLoss(
        weight=modifier_class_weights.to(device) if CONFIG["use_class_weights"] else None,
        label_smoothing=CONFIG.get("label_smoothing", 0.0))

    with torch.no_grad():
        for batch in loader:
            binary_labels   = batch.pop("binary_labels").to(device)
            modifier_labels = batch.pop("modifier_labels").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}

            binary_logits, modifier_logits = model(**batch)

            # Threshold-based binary prediction
            binary_probs_yes = torch.softmax(binary_logits, dim=-1)[:, yes_idx]
            binary_preds     = (binary_probs_yes >= binary_threshold).long()

            # Modifier loss on binary_label=yes rows only
            yes_mask = binary_labels == yes_idx
            mod_loss = (mce(modifier_logits[yes_mask], modifier_labels[yes_mask])
                        if yes_mask.any() else
                        torch.tensor(0.0, device=device))
            loss = (CONFIG["binary_loss_weight"] * bce(binary_logits, binary_labels)
                    + CONFIG["modifier_loss_weight"] * mod_loss)
            total_loss += loss.item()
            n_batches  += 1

            all_binary_true.extend(binary_labels.cpu().tolist())
            all_binary_pred.extend(binary_preds.cpu().tolist())
            all_binary_probs.extend(binary_probs_yes.cpu().tolist())
            all_modifier_true.extend(modifier_labels.cpu().tolist())
            all_modifier_pred.extend(modifier_logits.argmax(dim=-1).cpu().tolist())

    binary_acc       = accuracy_score(all_binary_true, all_binary_pred)
    binary_f1        = f1_score(all_binary_true, all_binary_pred, average="binary",
                                pos_label=yes_idx, zero_division=0)
    modifier_macro_f1 = f1_score(all_modifier_true, all_modifier_pred, average="macro",
                                  labels=list(range(len(CONFIG["modifier_classes"]))),
                                  zero_division=0)

    # Yes-conditioned modifier macro-F1 (matches task evaluation metric)
    yes_idx_eval = [i for i, b in enumerate(all_binary_true) if b == yes_idx]
    if yes_idx_eval:
        mod_true_yes = [all_modifier_true[i] for i in yes_idx_eval]
        mod_pred_yes = [all_modifier_pred[i] for i in yes_idx_eval]
        modifier_macro_f1_yes = f1_score(
            mod_true_yes, mod_pred_yes, average="macro",
            labels=list(range(len(CONFIG["modifier_classes"]))), zero_division=0)
    else:
        modifier_macro_f1_yes = 0.0

    return {
        "loss":                  total_loss / max(n_batches, 1),
        "binary_accuracy":       binary_acc,
        "binary_f1":             binary_f1,
        "modifier_macro_f1":     modifier_macro_f1,
        "modifier_macro_f1_yes": modifier_macro_f1_yes,
        "binary_true":    all_binary_true,  "binary_pred":    all_binary_pred,
        "binary_probs":   all_binary_probs,
        "modifier_true":  all_modifier_true, "modifier_pred":  all_modifier_pred,
    }


def find_optimal_binary_threshold(model, val_loader, device):
    """
    Grid-search for the binary threshold that maximises **macro-F1** on the
    validation set (i.e., the average of 'no' F1 and 'yes' F1).

    Motivation
    ----------
    The dataset is 72.5% 'yes', so models trained with cross-entropy tend
    to collapse 'no' recall. Lowering the threshold below 0.5 shifts the
    decision boundary to recover 'no' predictions without any retraining.
    Using macro-F1 as the search objective forces the chosen threshold to
    balance both classes.

    Scans thresholds in [0.10, 0.90] with step 0.05.

    Returns
    -------
    best_threshold : float
    best_macro_f1  : float (on the validation set)
    """
    model.eval()
    all_true, all_probs = [], []
    yes_idx = CONFIG["binary_classes"].index("yes")

    with torch.no_grad():
        for batch in val_loader:
            binary_labels = batch.pop("binary_labels").to(device)
            batch.pop("modifier_labels")
            batch = {k: v.to(device) for k, v in batch.items()}
            binary_logits, _ = model(**batch)
            probs = torch.softmax(binary_logits, dim=-1)[:, yes_idx]
            all_true.extend(binary_labels.cpu().tolist())
            all_probs.extend(probs.cpu().tolist())

    best_threshold, best_macro_f1 = 0.5, 0.0
    for t in [v / 100 for v in range(10, 91, 5)]:
        preds   = [1 if p >= t else 0 for p in all_probs]
        macro_f1 = f1_score(all_true, preds, average="macro",
                             labels=[0, 1], zero_division=0)
        if macro_f1 > best_macro_f1:
            best_macro_f1, best_threshold = macro_f1, t

    return best_threshold, best_macro_f1


def train_one_encoder(encoder_cfg, train_df, val_df, test_df):
    name, hf_id    = encoder_cfg["name"], encoder_cfg["hf_id"]
    use_roles      = CONFIG.get("use_speaker_roles", False)
    sep_line       = "=" * 70
    print(f"\n{sep_line}\nTraining {name} ({hf_id})"
          f"  [speaker_roles={'ON' if use_roles else 'OFF'}]\n{sep_line}")

    tokenizer  = AutoTokenizer.from_pretrained(hf_id)
    collate_fn = make_collate_fn(tokenizer, CONFIG["max_length"],
                                  use_speaker_roles=use_roles)

    train_ds = DonationIntentDataset(train_df)
    val_ds   = DonationIntentDataset(val_df)
    test_ds  = DonationIntentDataset(test_df)

    enc_batch_size = encoder_cfg.get("batch_size", CONFIG["batch_size"])
    if CONFIG["use_oversampling"]:
        sampler = make_oversampling_sampler(train_df)
        train_loader = DataLoader(train_ds, batch_size=enc_batch_size,
                                   sampler=sampler, collate_fn=collate_fn)
    else:
        train_loader = DataLoader(train_ds, batch_size=enc_batch_size,
                                   shuffle=True, collate_fn=collate_fn)
    val_loader  = DataLoader(val_ds,  batch_size=CONFIG["eval_batch_size"],
                              shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_ds, batch_size=CONFIG["eval_batch_size"],
                              shuffle=False, collate_fn=collate_fn)

    model = SpeakerAwareTwoHeadClassifier(
        hf_id,
        n_binary=len(CONFIG["binary_classes"]),
        n_modifier=len(CONFIG["modifier_classes"]),
        use_speaker_roles=use_roles,
    ).to(DEVICE)

    bce = nn.CrossEntropyLoss(
        weight=binary_class_weights.to(DEVICE) if CONFIG["use_class_weights"] else None,
        label_smoothing=CONFIG.get("label_smoothing", 0.0))
    mce = nn.CrossEntropyLoss(
        weight=modifier_class_weights.to(DEVICE) if CONFIG["use_class_weights"] else None,
        label_smoothing=CONFIG.get("label_smoothing", 0.0))

    encoder_lr = encoder_cfg.get("lr", CONFIG["lr"])
    optimizer  = torch.optim.AdamW(model.parameters(), lr=encoder_lr,
                                    weight_decay=CONFIG["weight_decay"], eps=1e-6)
    total_steps = len(train_loader) * CONFIG["num_epochs"]
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * CONFIG["warmup_ratio"]),
        num_training_steps=total_steps,
    )

    ckpt_path            = os.path.join(CONFIG["out_dir"], CONFIG["checkpoint_dir_name"],
                                        f"{name}_best.pt")
    best_val_score       = -1.0
    epochs_without_improve = 0
    history              = []

    for epoch in range(1, CONFIG["num_epochs"] + 1):
        model.train()
        running_loss = 0.0
        _n = CONFIG["num_epochs"]
        pbar = tqdm(train_loader, desc=f"[{name}] epoch {epoch}/{_n}")

        for batch in pbar:
            binary_labels   = batch.pop("binary_labels").to(DEVICE)
            modifier_labels = batch.pop("modifier_labels").to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            optimizer.zero_grad()
            binary_logits, modifier_logits = model(**batch)

            yes_mask = binary_labels == CONFIG["binary_classes"].index("yes")
            mod_loss = (mce(modifier_logits[yes_mask], modifier_labels[yes_mask])
                        if yes_mask.any() else
                        torch.tensor(0.0, device=DEVICE))
            loss = (CONFIG["binary_loss_weight"] * bce(binary_logits, binary_labels)
                    + CONFIG["modifier_loss_weight"] * mod_loss)

            if torch.isnan(loss):
                print("  [WARNING] NaN loss detected -- skipping batch.")
                optimizer.zero_grad()
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["max_grad_norm"])
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        # Early-stopping uses default threshold=0.5 for consistency with
        # previous experiments; threshold tuning happens AFTER training.
        val_metrics = evaluate(model, val_loader, DEVICE, binary_threshold=0.5)
        val_score   = (val_metrics["binary_f1"] + val_metrics["modifier_macro_f1_yes"]) / 2
        history.append({
            "epoch":                   epoch,
            "train_loss":              running_loss / len(train_loader),
            "val_loss":                val_metrics["loss"],
            "val_binary_f1":           val_metrics["binary_f1"],
            "val_binary_accuracy":     val_metrics["binary_accuracy"],
            "val_modifier_macro_f1":   val_metrics["modifier_macro_f1"],
            "val_modifier_macro_f1_yes": val_metrics["modifier_macro_f1_yes"],
        })
        _tl = history[-1]["train_loss"]
        _bf = val_metrics["binary_f1"]
        _mf = val_metrics["modifier_macro_f1_yes"]
        print(f"  epoch {epoch}: train_loss={_tl:.4f} "
              f"val_binary_f1={_bf:.4f} val_modifier_macroF1|yes={_mf:.4f}")

        if val_score > best_val_score:
            best_val_score       = val_score
            epochs_without_improve = 0
            torch.save(model.state_dict(), ckpt_path)
            print(f"  -> new best (avg F1={val_score:.4f}), checkpoint saved.")
        else:
            epochs_without_improve += 1
            if epochs_without_improve >= CONFIG["early_stopping_patience"]:
                print(f"  -> no improvement for {epochs_without_improve} epochs, stopping early.")
                break

    # -- Reload best checkpoint --------------------------------------------
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))

    # -- Threshold tuning on validation set --------------------------------
    # Scan P(yes) thresholds to maximise macro-F1 on val; apply to test.
    best_thresh, thresh_val_macro_f1 = find_optimal_binary_threshold(
        model, val_loader, DEVICE)
    print(f"\n  Threshold search -> best thresh={best_thresh:.2f}"
          f"  (val macro-F1={thresh_val_macro_f1:.4f})")

    # -- Final test evaluation ---------------------------------------------
    test_m_default = evaluate(model, test_loader, DEVICE, binary_threshold=0.5)
    test_m_tuned   = evaluate(model, test_loader, DEVICE, binary_threshold=best_thresh)

    yes_class_idx = CONFIG["binary_classes"].index("yes")
    for tm in (test_m_default, test_m_tuned):
        yes_list = [i for i, t in enumerate(tm["binary_true"]) if t == yes_class_idx]
        if yes_list:
            mt = [tm["modifier_true"][i] for i in yes_list]
            mp = [tm["modifier_pred"][i] for i in yes_list]
            tm["modifier_macro_f1_given_binary_yes"] = f1_score(
                mt, mp, average="macro",
                labels=list(range(len(CONFIG["modifier_classes"]))), zero_division=0)
        else:
            tm["modifier_macro_f1_given_binary_yes"] = 0.0

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "name":                              name,
        "hf_id":                             hf_id,
        "history":                           history,
        "test_metrics":                      test_m_tuned,       # primary result: tuned threshold
        "test_metrics_default":              test_m_default,     # secondary: argmax baseline
        "test_binary_accuracy":              test_m_tuned["binary_accuracy"],
        "test_binary_f1":                    test_m_tuned["binary_f1"],
        "test_modifier_macro_f1":            test_m_tuned["modifier_macro_f1"],
        "test_modifier_macro_f1_given_yes":  test_m_tuned["modifier_macro_f1_given_binary_yes"],
        "best_threshold":                    best_thresh,
        "ckpt_path":                         ckpt_path,
    }


## 8. Run training for both encoders

Trains RoBERTa-base then DeBERTa-v3-base sequentially (each fully unloaded
from GPU memory before the next starts, same discipline as the LLM-annotator
notebook's model-swap loop).

In [ ]:
# =============================================================
# 8. RUN BOTH ENCODERS
# =============================================================
results = {}
for encoder_cfg in CONFIG["encoders"]:
    results[encoder_cfg["name"]] = train_one_encoder(encoder_cfg, train_df, val_df, test_df)

print("\nDone training all encoders:", list(results.keys()))


In [ ]:
# =============================================================
# 8b. THRESHOLD COMPARISON TABLE
# =============================================================
# Compare default (argmax at 0.5) vs tuned threshold per encoder.
print(f"\n{'='*70}")
print("Binary head: default threshold (0.5) vs val-tuned threshold")
print(f"{'='*70}")
header = f"{'Model':<20} {'Thresh':>6}  {'Binary Acc':>10}  {'Binary F1':>10}  "
header += f"{'no F1':>8}  {'yes F1':>8}  {'Mod F1|yes':>11}"
print(header)
print("-" * len(header))

for name, r in results.items():
    for label, tm, thresh in [
        ("(default)", r["test_metrics_default"], 0.5),
        ("(tuned)",   r["test_metrics"],         r["best_threshold"]),
    ]:
        from sklearn.metrics import f1_score as _f1
        no_f1  = _f1(tm["binary_true"], tm["binary_pred"], pos_label=0,
                      average="binary", zero_division=0)
        yes_f1 = _f1(tm["binary_true"], tm["binary_pred"], pos_label=1,
                      average="binary", zero_division=0)
        row = (f"{name+' '+label:<20} {thresh:>6.2f}  "
               f"{tm['binary_accuracy']:>10.4f}  {tm['binary_f1']:>10.4f}  "
               f"{no_f1:>8.4f}  {yes_f1:>8.4f}  "
               f"{tm['modifier_macro_f1_given_binary_yes']:>11.4f}")
        print(row)
    print()


## 9. Test-set majority-class baseline (matched to the actual test split)

The section-2b baseline was computed over the whole dataset for a quick look;
this is the baseline actually reported against the trained models, computed
from the **train split's** majority class and evaluated on the **test
split**, which is the fair comparison.

In [ ]:
# =============================================================
# 9. TEST-SET MAJORITY-CLASS BASELINE
# =============================================================
train_majority_binary = Counter(train_df["binary_id"]).most_common(1)[0][0]
train_majority_modifier = Counter(train_df["modifier_id"]).most_common(1)[0][0]

test_binary_true = test_df["binary_id"].values
test_modifier_true = test_df["modifier_id"].values

baseline_test_binary_pred = [train_majority_binary] * len(test_df)
baseline_test_modifier_pred = [train_majority_modifier] * len(test_df)

# Restrict modifier baseline to binary_label="yes" test rows to match the
# conditioned evaluation metric used for trained models (modifier_macro_f1_given_binary_yes).
test_yes_mask = [b == CONFIG["binary_classes"].index("yes") for b in test_binary_true]
test_modifier_true_yes   = [test_modifier_true[i]          for i, m in enumerate(test_yes_mask) if m]
baseline_modifier_pred_yes = [baseline_test_modifier_pred[i] for i, m in enumerate(test_yes_mask) if m]

baseline_row = {
    "model": "majority_baseline",
    "binary_accuracy": accuracy_score(test_binary_true, baseline_test_binary_pred),
    "binary_f1": f1_score(test_binary_true, baseline_test_binary_pred, average="binary",
                           pos_label=CONFIG["binary_classes"].index("yes"), zero_division=0),
    "modifier_macro_f1": f1_score(test_modifier_true, baseline_test_modifier_pred, average="macro",
                                   labels=list(range(len(CONFIG["modifier_classes"]))), zero_division=0),
    # Conditioned on binary_label="yes" -- matches the metric reported for trained models.
    "modifier_macro_f1_given_binary_yes": f1_score(
        test_modifier_true_yes, baseline_modifier_pred_yes,
        average="macro",
        labels=list(range(len(CONFIG["modifier_classes"]))), zero_division=0),
}
print("Majority-class baseline on TEST split:")
print(json.dumps(baseline_row, indent=2))


## 10. Results summary table — both encoders vs. majority baseline

In [ ]:
# =============================================================
# 10. RESULTS SUMMARY
# =============================================================
summary_rows = [baseline_row]
for name, r in results.items():
    summary_rows.append({
        "model": name,
        "binary_accuracy": r["test_binary_accuracy"],
        "binary_f1": r["test_binary_f1"],
        "modifier_macro_f1": r["test_modifier_macro_f1"],
        "modifier_macro_f1_given_binary_yes": r["test_modifier_macro_f1_given_yes"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(CONFIG["out_dir"], "classifier_test_results.csv"), index=False)
display(summary_df)


### 10a. Bar chart — accuracy/F1 comparison across encoders + baseline

In [ ]:
plot_df = summary_df.melt(
    id_vars="model",
    value_vars=["binary_accuracy", "binary_f1", "modifier_macro_f1"],
    var_name="metric", value_name="score",
)
plt.figure(figsize=(9, 5))
sns.barplot(data=plot_df, x="metric", y="score", hue="model")
plt.ylim(0, 1)
plt.title("Classifier performance vs. majority-class baseline (test split)")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["out_dir"], "classifier_comparison_bar.png"), dpi=150)
plt.show()


### 10b. Confusion matrices — binary_label, per encoder

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(5.5 * len(results), 4.5))
if len(results) == 1:
    axes = [axes]
for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(r["test_metrics"]["binary_true"], r["test_metrics"]["binary_pred"],
                           labels=list(range(len(CONFIG["binary_classes"]))))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CONFIG["binary_classes"], yticklabels=CONFIG["binary_classes"], ax=ax)
    ax.set_title(f"{name} -- binary_label")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["out_dir"], "confusion_matrix_binary.png"), dpi=150)
plt.show()


### 10c. Confusion matrices — modifier, per encoder

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(5.5 * len(results), 4.5))
if len(results) == 1:
    axes = [axes]
for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(r["test_metrics"]["modifier_true"], r["test_metrics"]["modifier_pred"],
                           labels=list(range(len(CONFIG["modifier_classes"]))))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges",
                xticklabels=CONFIG["modifier_classes"], yticklabels=CONFIG["modifier_classes"], ax=ax)
    ax.set_title(f"{name} -- modifier")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["out_dir"], "confusion_matrix_modifier.png"), dpi=150)
plt.show()


### 10d. Training curves — val F1 per epoch, per encoder

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name, r in results.items():
    hist = pd.DataFrame(r["history"])
    axes[0].plot(hist["epoch"], hist["val_binary_f1"], marker="o", label=name)
    axes[1].plot(hist["epoch"], hist["val_modifier_macro_f1"], marker="o", label=name)
axes[0].set_title("Validation binary_label F1"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].set_title("Validation modifier macro-F1"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["out_dir"], "training_curves.png"), dpi=150)
plt.show()


## 10e. Classification reports (full precision/recall/F1 per class)

In [ ]:
for name, r in results.items():
    _sep = "=" * 70
    print(f"\n{_sep}\n{name} -- binary_label classification report\n{_sep}")
    print(classification_report(r["test_metrics"]["binary_true"], r["test_metrics"]["binary_pred"],
                                 target_names=CONFIG["binary_classes"], zero_division=0))
    print(f"{name} -- modifier classification report")
    print(classification_report(r["test_metrics"]["modifier_true"], r["test_metrics"]["modifier_pred"],
                                 target_names=CONFIG["modifier_classes"], zero_division=0))


## 11. Summary

Output files written to `CONFIG["out_dir"]`:

- `classifier_test_results.csv` — accuracy/F1 for both encoders + majority baseline, on the held-out test split.
- `checkpoints/<encoder-name>_best.pt` — best-val-F1 model weights for each encoder (state_dict, reloadable via `TwoHeadIntentClassifier` + `load_state_dict`).
- `label_distribution.png` — data diagnostics (section 2b).
- `classifier_comparison_bar.png`, `confusion_matrix_binary.png`, `confusion_matrix_modifier.png`, `training_curves.png` — result charts.

**Notes on this run's dataset:**
1. The 1017-row dataset has no partition/annotator column, so there is no partition-based stratification and no partition-specific drift check (unlike the earlier 5-annotator design) -- splitting is stratified by the `binary_label`×`modifier` combination only.
2. Blank/NaN `modifier` values were filled to `"none"`.
3. `modifier` head loss is masked to `binary_label == "yes"` rows only (both during backprop and in the class-weight computation) -- the model still outputs modifier logits for every row for a clean fixed-shape two-head classifier, but "no" rows never contribute to the modifier loss, avoiding the train/eval distribution mismatch that under-weighted rare modifier classes in earlier runs. Both the unconditional (`modifier_macro_f1`) and conditioned (`modifier_macro_f1_given_binary_yes`) metrics are reported in section 10.
4. Training rows for `binary=no`, `modifier=deferred`, and `modifier=conditional` are augmented via label-safety-guarded EDA (section 3b) -- these are the classes that were structurally starved of examples and previously either failed to learn (`deferred`) or were entirely unlearnable (`conditional`, ~9-13 real train examples). Augmentation is applied strictly post-split, train-only.
5. Both classification heads use label smoothing (`CONFIG["label_smoothing"] = 0.1`) to reduce the majority-class overconfidence collapse observed in the previous run (DeBERTa's modifier head predicted "none" for 100% of test rows).
